In [6]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from IPython.display import display, Markdown
import pandas as pd
import time


In [2]:
def load_model():
    model_name = "Qwen/Qwen3-8B"
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        torch_dtype="auto",
        device_map="auto",
        local_files_only=True #swith it off to download from hugging face if you don't have it locally
    )
    tokenizer = AutoTokenizer.from_pretrained(model_name, local_files_only=True)
    return {"model": model, "tokenizer": tokenizer}


In [3]:
model_and_tokenizer = load_model()
model = model_and_tokenizer["model"]
tokenizer = model_and_tokenizer["tokenizer"]

Loading weights: 100%|██████████| 399/399 [00:02<00:00, 197.95it/s]


In [4]:
params = {
    "model": model,
    "tokenizer": tokenizer,
    "web_query_max_new_t": 50,
    "titles_limit": 5,
    "top_k_titles": 5,
    "chunk_size": 200,
    "chunk_overlap_fraction": 0.2,
    "top_k_chunks": 5,
    "answer_max_new_t": 500
}

from rag_flow import RAGFlow
rag = RAGFlow(params)

answer = rag.run("How does retrieval-augmented generation reduce hallucinations in large language models??")


2026-10-05 18:33:55,272 | INFO | No device provided, using mps
2026-10-05 18:33:55,428 | INFO | HTTP Request: HEAD https://huggingface.co/cross-encoder/ms-marco-MiniLM-L6-v2/resolve/main/modules.json "HTTP/1.1 404 Not Found"
2026-10-05 18:33:55,431 | INFO | No modules.json found for cross-encoder/ms-marco-MiniLM-L6-v2, initializing a new CrossEncoder model.
2026-10-05 18:33:55,517 | INFO | HTTP Request: GET https://huggingface.co/api/models/cross-encoder/ms-marco-MiniLM-L6-v2 "HTTP/1.1 200 OK"
2026-10-05 18:33:55,609 | INFO | HTTP Request: GET https://huggingface.co/api/models/cross-encoder/ms-marco-MiniLM-L6-v2 "HTTP/1.1 200 OK"
2026-10-05 18:33:55,696 | INFO | HTTP Request: HEAD https://huggingface.co/cross-encoder/ms-marco-MiniLM-L6-v2/resolve/main/adapter_config.json "HTTP/1.1 404 Not Found"
2026-10-05 18:33:55,789 | INFO | HTTP Request: GET https://huggingface.co/api/models/cross-encoder/ms-marco-MiniLM-L6-v2 "HTTP/1.1 200 OK"
2026-10-05 18:33:55,878 | INFO | HTTP Request: GET htt

"retrieval-augmented generation hallucinations large language models"


Batches: 100%|██████████| 12/12 [00:01<00:00, 10.69it/s]
2026-10-05 18:34:06,643 | INFO | Answer generation input tokens: 2253


In [5]:
answer

'Retrieval-Augmented Generation (RAG) reduces hallucinations in large language models by grounding responses in retrieved evidence from external sources. This allows models to use up-to-date and verifiable information beyond their fixed parameters, thereby mitigating the risk of generating false or misleading information. \n\nThe context indicates that while RAG significantly reduces hallucinations, reliability remains a concern when the retrieved content is irrelevant, outdated, or misleading, as inaccuracies in retrieval can propagate into the final response. \n\nURL: https://arxiv.org/pdf/2610.01936'

## Evaluation
### System level:
- Token usage per prompt.
- End-to-End Latency
- Context recall (relevant answers / (relevant answers + irrelevant answers). To determine whether an answer is relevant, I will use a threshold of similarity between the actual answer and the correct answer. For example, the cosine similarity is 0.9 then the answer is marked relevant.
- Context precision (relevant answers / total answers)
- Faithfulness (Is the answer strictly true to the retrieved context?" (Checks for hallucination) I am going to use RAGAS library ( if I have no issues configuring it and a LLM-as-judge)
### Retrieval level
- Context precision/ recall @ k
- Mean reciprocal rank (MRR)
- Normalized Discounted Cumulative Gain (NDCG)  how well top-k is sorted




## To do:
- generate ground truth
- test and record the results
- download 1 -2 more models and test the same flow with them. Chose the one that performed best
- With best model, fine-tune the system:
    - test different numbers of searched titles
    - test different number of downloaded articles
    - try different chunk sizes
    - try different number of top k in the chunk ranker
    - try a set of promot augmentation.

In [ ]:
sys_evaluation_df = {
    "mean_latency": "",
    "mean_token_usage":"",
    "context_recall":"",
    "context_precision":"",
    "mean_faithfulness":""
}

In [7]:
golden_df = pd.read_csv("golden_dataset.csv")



In [25]:
results = []

for _, row in golden_df.iterrows():

    golden_id = row["golden_id"]
    question = row["question"]
    print(golden_id)
    start = time.perf_counter()
    answer = flow(model, tokenizer, question)
    latency = time.perf_counter() - start

    results.append({
        "golden_id": golden_id,
        "answer": answer,
        "latency": latency
    })

evaluation_df = pd.DataFrame(results)
evaluation_df.to_csv("system_eval_results.csv")


1
Input tokens: 64


Loading weights: 100%|██████████| 105/105 [00:00<00:00, 12876.87it/s]


2 k
{'article_url': 'https://arxiv.org/pdf/2610.02926', 'title': 'How To Train Your World Model: Fine-tuning vs RAG for LM-based World Modeling', 'chunk_index': 17, 'text': 'single letter tokens removes the generation step that makes script-switching possible. This makes TruthfulQA MC accuracy an unreliable signal in either direction under distractor conditions, a methodological point for future benchmark design. 5 Table 2: TruthfulQA accuracy: clean baseline and worst distractor condition. 95% CIs via bootstrap (n=2000). ns: not significant; ∗∗p<0.01; ∗∗∗p<0.001. Model Clean Worst ∆p Llama-3.1-8B 0.510 0.478 (wiki) −0.032 ns Llama-3.2-3B 0.554 0.454 (zh_l) −0.100 ∗∗∗ Phi-3-mini 0.652 0.616 (wiki) −0.036 ns Mistral-7B 0.678 0.626 (wiki) −0.052 ∗∗ Qwen-2.5-3B 0.440 0.440 (es_l) +0.000 ns Two observations in Table 2 warrant brief note. First, Phi-3-mini and Qwen-2.5-3B show small positive accuracy deltas under Hindi and Chinese distractors (e.g., Phi-3-mini reaches 0.770 under Hindi, ∆= 

Loading weights: 100%|██████████| 105/105 [00:00<00:00, 14599.77it/s]


2 k
{'article_url': 'https://arxiv.org/pdf/2610.02926', 'title': 'How To Train Your World Model: Fine-tuning vs RAG for LM-based World Modeling', 'chunk_index': 7, 'text': '3. We characterize a failure mode taxonomy: script-switching hallucination, abstention escalation, and stability, showing that the operative mode is determined by model-specific spontaneous generative capacity in the distractor language. The clean dissociation across five models with identical inputs is itself the finding: it rules out general prompt sensitivity and points to a tractable, model-level mechanism. 2 2 Related Work Multilingual evaluation. Established multilingual benchmarks such as XTREME [Hu et al., 2020] and XGLUE [Liang et al., 2020] evaluate models on tasks presented in a target language, measuring cross-lingual transfer and per-language performance. Open-domain multilingual QA benchmarks such as MKQA [Longpre et al., 2021] similarly assume that task language and prompt language are aligned. Our se

...

2 k
{'article_url': 'https://arxiv.org/pdf/2610.02926', 'title': 'How To Train Your World Model: Fine-tuning vs RAG for LM-based World Modeling', 'chunk_index': 7, 'text': '3. We characterize a failure mode taxonomy: script-switching hallucination, abstention escalation, and stability, showing that the operative mode is determined by model-specific spontaneous generative capacity in the distractor language. The clean dissociation across five models with identical inputs is itself the finding: it rules out general prompt sensitivity and points to a tractable, model-level mechanism. 2 2 Related Work Multilingual evaluation. Established multilingual benchmarks such as XTREME [Hu et al., 2020] and XGLUE [Liang et al., 2020] evaluate models on tasks presented in a target language, measuring cross-lingual transfer and per-language performance. Open-domain multilingual QA benchmarks such as MKQA [Longpre et al., 2021] similarly assume that task language and prompt language are aligned. Our se

Loading weights: 100%|██████████| 105/105 [00:00<00:00, 13245.97it/s]


2 k
{'article_url': 'https://arxiv.org/pdf/2610.02926', 'title': 'How To Train Your World Model: Fine-tuning vs RAG for LM-based World Modeling', 'chunk_index': 17, 'text': 'single letter tokens removes the generation step that makes script-switching possible. This makes TruthfulQA MC accuracy an unreliable signal in either direction under distractor conditions, a methodological point for future benchmark design. 5 Table 2: TruthfulQA accuracy: clean baseline and worst distractor condition. 95% CIs via bootstrap (n=2000). ns: not significant; ∗∗p<0.01; ∗∗∗p<0.001. Model Clean Worst ∆p Llama-3.1-8B 0.510 0.478 (wiki) −0.032 ns Llama-3.2-3B 0.554 0.454 (zh_l) −0.100 ∗∗∗ Phi-3-mini 0.652 0.616 (wiki) −0.036 ns Mistral-7B 0.678 0.626 (wiki) −0.052 ∗∗ Qwen-2.5-3B 0.440 0.440 (es_l) +0.000 ns Two observations in Table 2 warrant brief note. First, Phi-3-mini and Qwen-2.5-3B show small positive accuracy deltas under Hindi and Chinese distractors (e.g., Phi-3-mini reaches 0.770 under Hindi, ∆= 

Loading weights: 100%|██████████| 105/105 [00:00<00:00, 13050.11it/s]


2 k
{'article_url': 'https://arxiv.org/pdf/2610.02926', 'title': 'How To Train Your World Model: Fine-tuning vs RAG for LM-based World Modeling', 'chunk_index': 1, 'text': 'Output Language Confusion under Multilingual Prompt Contamination Riju Marwah1,2, Ritvik Garimella2, Khusham Bansal2,3, Atishay Jain2,4, Amit Sheth2,5 1University of Tübingen, Germany 2Artificial Intelligence Institute, University of South Carolina, USA 3Thapar Institute of Engineering & Technology, India 4Indian Institute of Technology Kanpur, India 5Indian AI Research Organization, India riju.marwah@student.uni-tuebingen.de Abstract Standard factual benchmarks assume clean monolingual prompts and exact-match scoring, two assumptions that break simultaneously in real-world multilingual de- ployment, from retrieval-augmented generation pipelines returning mixed-language passages to users pasting multilingual web content. We introduce Multilingual Dis- tractor Interference (MDI), a lightweight and fully replicable ev

In [ ]:
import pandas as pd

rag_df = pd.DataFrame([
    {
        "question": "What is Retrieval-Augmented Generation?",
        "answer": "RAG combines information retrieval with language generation."
    },
    {
        "question": "What is a CrossEncoder used for?",
        "answer": "A CrossEncoder can rerank retrieved documents based on their relevance to a query."
    },
    {
        "question": "What is semantic search?",
        "answer": "Semantic search retrieves information based on meaning rather than exact keyword matches."
    }
])

rag_df

2026-10-05 17:35:48,231 | INFO | No device provided, using mps
2026-10-05 17:35:48,404 | INFO | HTTP Request: HEAD https://huggingface.co/cross-encoder/ms-marco-MiniLM-L6-v2/resolve/main/modules.json "HTTP/1.1 404 Not Found"
2026-10-05 17:35:48,406 | INFO | No modules.json found for cross-encoder/ms-marco-MiniLM-L6-v2, initializing a new CrossEncoder model.
2026-10-05 17:35:48,494 | INFO | HTTP Request: GET https://huggingface.co/api/models/cross-encoder/ms-marco-MiniLM-L6-v2 "HTTP/1.1 200 OK"
2026-10-05 17:35:48,652 | INFO | HTTP Request: GET https://huggingface.co/api/models/cross-encoder/ms-marco-MiniLM-L6-v2 "HTTP/1.1 200 OK"
2026-10-05 17:35:48,736 | INFO | HTTP Request: HEAD https://huggingface.co/cross-encoder/ms-marco-MiniLM-L6-v2/resolve/main/adapter_config.json "HTTP/1.1 404 Not Found"
2026-10-05 17:35:48,826 | INFO | HTTP Request: GET https://huggingface.co/api/models/cross-encoder/ms-marco-MiniLM-L6-v2 "HTTP/1.1 200 OK"
2026-10-05 17:35:48,912 | INFO | HTTP Request: GET htt

KeyError: 'top_k_titles'